In [12]:
# imports
import json

import logging

# Set up logging configuration at the top of your notebook or script
logging.basicConfig(
    level=logging.INFO,  # Change to DEBUG for more verbosity
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-32B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
   model_name,
   dtype="auto",
   device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

2025-10-12 07:38:23,006 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

2025-10-12 07:38:25,515 - WARNING - Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
def generate_for_prompt(model, tokenizer, user_prompt, **kwargs):
    def build_messages(user_prompt):
        return [
            {
                "role": "system",
                "content": (
                    "You are an expert assistant specialized in generating accurate and high-quality Java code from natural language descriptions. "
                    "When responding, output only the Java code, with no explanations or comments. "
                    "Do not change or reformat code in the prompt; just continue and return the full code (prompt plus your completion) as one complete Java code block."
                )
            },
            {"role": "user", "content": user_prompt}
        ]

    max_new_tokens = kwargs.get("max_new_tokens", 1024)
    top_p = kwargs.get("top_p", 0.95)
    temperature = kwargs.get("temperature", 0.1)
    top_k = kwargs.get("top_k", 0)

    messages = build_messages(user_prompt)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        # top_p=top_p,
        # temperature=temperature,
        # top_k=top_k
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True)[0]
    return response

In [ ]:
from tqdm import tqdm
import math
from concurrent.futures import ThreadPoolExecutor, as_completed


def generate_for_prompts_v2(model, tokenizer, prompts, chunk_size=4, max_workers=4, **kwargs):
    """
    Generate completions for prompts in parallel, divided into chunks.
    Reports overall progress across all prompts using tqdm.
    """
    total = len(prompts)
    completions = [None] * total
    num_chunks = math.ceil(total / chunk_size)
    logging.info(
        f"Total prompts: {total}, Chunk size: {chunk_size}, Chunks: {num_chunks}")

    def process_chunk(chunk_prompts, chunk_indices):
        chunk_results = []
        for idx, prompt in zip(chunk_indices, chunk_prompts):
            response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
            chunk_results.append((idx, response))
        return chunk_results

    # Prepare chunks
    chunks = [
        (prompts[i:i+chunk_size], list(range(i, min(i+chunk_size, total))))
        for i in range(0, total, chunk_size)
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor, tqdm(total=total, desc="Overall Progress") as pbar:
        futures = {executor.submit(process_chunk, chunk_prompts, chunk_indices): (
            chunk_prompts, chunk_indices) for chunk_prompts, chunk_indices in chunks}
        for future in as_completed(futures):
            chunk_results = future.result()
            for idx, response in chunk_results:
                completions[idx] = response
                pbar.update(1)

    logging.info("All completions finished.")
    return completions

In [15]:
def generate_for_prompts(model, tokenizer, prompts, **kwargs):
    completions = []
    for idx, prompt in enumerate(prompts):
        print(f"Generating completion for problem {idx+1}/{len(prompts)}")
        response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
        completions.append(response if isinstance(
            response, list) else [response])

        print(f'Completion for problem {idx+1}:\n{response}\n{"-"*40}\n')

    return completions

In [ ]:
def generate_for_dataset(model, tokenizer, problems, **kwargs):
    user_prompts = [problem['prompt'] for problem in problems]
    
    completions = generate_for_prompts_v2(model, tokenizer, user_prompts, **kwargs) if kwargs.get(
        'parallel', False) else generate_for_prompts(model, tokenizer, user_prompts, **kwargs)
    
    # Save completions to a JSON file
    output_path = 'completions.json'
    with open(output_path, 'w') as f:
        json.dump(completions, f, indent=4)

    return completions

In [17]:
test_prompt = """
import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        and integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {        
""";

In [18]:
print(generate_for_prompt(model, tokenizer, test_prompt))

import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        and integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {        
        return number - Math.floor(number);
    }
}


In [ ]:
import os
print(os.getcwd())
ds_json_path = os.path.join('../../../../../..', 'benchmark/datasets/humaneval-x/humanevalx-java-refined.json')

problems = json.load(open(ds_json_path, 'r'))
logging.info(f'Loaded {len(problems)} problems from "{ds_json_path}"')

generate_for_dataset(model, tokenizer, problems, parallel=true, chunk_size=2, max_workers=4)

2025-10-12 07:45:53,649 - INFO - Loaded 164 problems from "../../../../../../benchmark/datasets/humaneval-x/humanevalx-java-refined.json"


/workspace/bigcode-evaluation-harness/benchmark/Qwen2.5-Coder-32B-Instruct/java/improve/pass@1/chat
Generating completion for problem 1/164
